# 02. Attaching an LLM and Talking to an Agent

Notebook 01 stood the dashboard up. This one covers the two things you actually *do* with a running `arcui` instance: wire an LLM's module stack in for REST introspection (`attach_llm`), and open a live conversation with an agent (`/ws/chat/{agent_id}`) or watch the team-flow stream (`/ws/team`).

**A note up front (SPEC-026 FR-5).** An earlier version of this notebook (and of `arcui`) described a live event-streaming pipeline — agents pushed telemetry over `/ws`, an `EventBuffer` batched it, a `RollingAggregator` computed rolling stats, a `SubscriptionManager` filtered per browser tab, and a three-token model gated a dedicated agent-push channel. **All of that was torn out.** `arcui` is now a read-only consumer of the durable arcstore record (`Observe`, covered in notebook 01) plus two WebSocket routes: `/ws/chat/{agent_id}` (bidirectional interactive chat) and `/ws/team` (read-only team-flow stream, with a narrow post-forward path). This notebook covers what's actually there today.

**You will learn:**

- What `attach_llm` really does — module-stack discovery for REST introspection, nothing more
- The real `/ws/chat/{agent_id}` protocol: auth handshake, `ready` frame, message frames, reconnect replay via `?since_seq=`
- The real `/ws/team` protocol: channel-scoped subscription, drop-oldest backpressure, human post-forwarding
- Where `arcllm`'s module stack (`CircuitBreakerModule`, `TelemetryModule`, `QueueModule`) surfaces once attached
- Why the shared `authenticate_ws` helper enforces the same auth rules on both WebSocket routes

## 1. Setup

Run the standard Arc walkthrough boilerplate. It locates the repo root, puts every package's ``src/`` and ``tests/`` directories on ``sys.path``, and loads ``.env`` if one exists. Every notebook in this set starts the same way so cells are interchangeable across packages.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

This notebook is fully **mock-first** — there is no live LLM, no real WebSocket client, and no API keys required. We use Starlette's :class:`TestClient`, which supports synchronous WebSocket interactions over an in-process ASGI transport.

In [2]:
import json
from unittest.mock import MagicMock

from starlette.testclient import TestClient

from arcui import attach_llm, create_app
from arcui.auth import AuthConfig

print("arcui imports loaded")

arcui imports loaded


/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/ipykernel_27063/11220891.py:4: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient


## 2. Why live telemetry — operational visibility

Federal deployments care about three things at once: *what happened*, *who did it*, and *can we act on it right now?* The first two are handled by the durable record (arcstore) and the audit trail. The third splits into two real, narrow surfaces in `arcui` today:

- **`attach_llm`** — walks an `arcllm` provider's module stack once at startup and exposes whichever modules it finds (`CircuitBreakerModule`, `TelemetryModule`, `QueueModule`) on `app.state` so REST routes like `/api/circuit-breakers`, `/api/budget`, `/api/queue` can report live module state on request. There is no event push, no buffering, no aggregation — it's a one-time discovery walk.
- **`/ws/chat/{agent_id}`** — the actual "act on it right now" surface: a human opens a live, bidirectional conversation with a specific running agent through `arcgateway`'s web adapter.

Neither of these is what the old notebook called "live telemetry." Rolling stats and cost dashboards now come from `Observe` querying the durable record on each request (notebook 01) — there's nothing to "attach" for that; it's just there.

In [3]:
# A 'why' demo — what an operator can do with each surface.
chat_view = {
    "when": "human wants to steer/query one running agent right now",
    "channel": "/ws/chat/{agent_id} — bidirectional",
    "reaction": "type a message, get a reply, redirect the agent",
}
observe_view = {
    "when": "human wants cost/trace/stats history",
    "channel": "REST over app.state.observe — on-demand query",
    "reaction": "filter by agent/time, no live push needed",
}
for k in chat_view:
    print(f"{k:8s}  chat={chat_view[k]!r:55s}  observe={observe_view[k]!r}")

when      chat='human wants to steer/query one running agent right now'  observe='human wants cost/trace/stats history'
channel   chat='/ws/chat/{agent_id} — bidirectional'                    observe='REST over app.state.observe — on-demand query'
reaction  chat='type a message, get a reply, redirect the agent'        observe='filter by agent/time, no live push needed'


The two surfaces are complementary and neither is the audit trail. `/ws/chat` is for a human talking to one agent right now. `Observe`-backed REST is for looking at what already happened, on demand. The durable arcstore record underneath both is what an auditor reads later.

## 3. `attach_llm` — module-stack discovery

Read the function once (`arcui/server.py`); it's short and it does exactly one thing:

Walk the LLM's module stack (`instance`, then `instance._inner`, then `_inner._inner`, ...) and, for each `CircuitBreakerModule` / `TelemetryModule` / `QueueModule` found, append it to the matching `app.state` list (`circuit_breakers`, `telemetry_modules`, `queue_modules`). That's it — no event callback is registered, no buffer is created, nothing is aggregated. The `label` parameter is accepted for call-site compatibility with the CLI but is otherwise unused (per the docstring).

Those lists are what `/api/circuit-breakers`, `/api/budget`, and `/api/queue` read from to report live in-process module state — e.g. whether a circuit breaker is currently open, or how many requests are queued. `attach_llm` is best called once per app at startup; calling it again just appends more module references to the same lists.

In [4]:
app = create_app(auth_config=AuthConfig({"viewer_token": "v", "operator_token": "o"}))

# A minimal stand-in for an LLM provider. The real shape is an arcllm
# adapter + module stack chained via _inner. A bare MagicMock with
# _inner=None is a one-module "stack" that terminates the walk immediately.
fake_llm = MagicMock()
fake_llm._inner = None  # end of stack

attach_llm(app, fake_llm, label="analyst-1")

# Nothing matched CircuitBreakerModule/TelemetryModule/QueueModule (a bare
# MagicMock isn't an instance of any of them), so all three lists stay empty.
print("circuit breakers found:", len(app.state.circuit_breakers))
print("telemetry modules found:", len(app.state.telemetry_modules))
print("queue modules found:", len(app.state.queue_modules))

circuit breakers found: 0
telemetry modules found: 0
queue modules found: 0


`label` is accepted for call-site compatibility with the CLI (which passes the agent's label at attach time) but `attach_llm` itself doesn't use it — there's no event to stamp it onto. Calling `attach_llm` twice on the same app with two different LLM instances just appends to the same three lists; each attached module shows up once per call.

An attached LLM with a stacked ``CircuitBreakerModule`` shows the discovery in action. We construct a tiny synthetic stack — adapter wrapped by a circuit breaker wrapped by a telemetry module — and attach it. The discovery walk records both modules, leaving the actual adapter alone because nothing in arcui needs to know about it.

In [5]:
from arcllm.modules.circuit_breaker import CircuitBreakerModule
from arcllm.modules.telemetry import TelemetryModule

fresh_app = create_app()

adapter = MagicMock()
adapter._inner = None

cb = MagicMock(spec=CircuitBreakerModule)
cb._inner = adapter

outer = MagicMock(spec=TelemetryModule)
outer._inner = cb

attach_llm(fresh_app, outer, label="full-stack-agent")

print("circuit_breakers:", len(fresh_app.state.circuit_breakers))
print("telemetry_modules:", len(fresh_app.state.telemetry_modules))

circuit_breakers: 1
telemetry_modules: 1


The walk uses ``getattr(current, '_inner', None)`` so a stack that ends in a bare adapter (no ``_inner`` attribute on the leaf) terminates cleanly without raising. This is also why ``attach_llm`` works with a plain ``MagicMock`` — as long as ``_inner`` returns ``None`` eventually, the walk terminates.

## 4. WebSocket protocols — `/ws/chat/{agent_id}` and `/ws/team`

Both WebSocket routes share the same first-message auth handshake (`arcui.ws_helpers.authenticate_ws`) and then diverge into route-specific framing.

**Shared auth handshake (both routes)**

- Client sends `{"token": "<bearer>"}` within 5 s (`AUTH_TIMEOUT_SECONDS`) of connecting.
- Server validates against `app.state.auth_config`. Only `viewer`/`operator` roles may proceed.
- Failure → one `{"error": "..."}` frame, then close: `4001` (timeout/malformed), `4003` (invalid token).

**`/ws/chat/{agent_id}` — after auth**

- Server resolves `agent_id` against the fleet roster; unknown agent → `4404`.
- Server registers the socket with `arcgateway`'s `WebPlatformAdapter`; at capacity → `4429`.
- Server sends `{"type": "ready", "chat_id": "<session_key>"}`.
- Client sends `{"type": "message", "text": "...", "client_seq": N}` frames; each is forwarded to `adapter.ingest(...)`.
- A reconnecting client can pass `?since_seq=N` in the connect URL to replay any frames after the highest `seq` it already saw.

**`/ws/team` — after auth**

- Server sends `{"type": "ready"}`.
- Server drains `app.state.team_stream` (a `TeamStreamHub`) and pushes rendered team-flow frames (`{"type": "team_message", "channel": ..., "from": ..., "body": ..., "mentions": [...], ...}`) — handles only, never raw DIDs.
- Client can post to a channel with `{"type": "post", "channel": "...", "text": "..."}`; the route forwards it to `app.state.team_post_forwarder`, which signs and routes it as the human entity. `arcui` never signs anything itself.
- `?channel=<name>` on the connect URL scopes the socket to one channel; omitted, the socket sees every channel.

There is no `event_batch`, no `subscribe` message type, no `ping` heartbeat frame, and no typed `UIEvent` broadcast anymore — those belonged to the deleted push pipeline. Every message on both routes is plain JSON.

In [6]:
# Stand up an app and walk the /ws/team handshake using TestClient.
# team_stream is always constructed by create_app() (unlike web_adapter,
# which needs gateway_config + team_root), so /ws/team works end-to-end
# even on a minimal app.
auth = AuthConfig({"viewer_token": "walk-viewer", "operator_token": "walk-op"})
app2 = create_app(auth_config=auth)
client = TestClient(app2)

with client.websocket_connect("/ws/team") as ws:
    ws.send_text(json.dumps({"token": "walk-viewer"}))
    ready = ws.receive_json()
    print("first server frame:", ready)

first server frame: {'type': 'ready'}


A quick REST-side check first: `/ws/chat/{agent_id}` needs a `web_adapter` on `app.state`, which `create_app()` only wires up when both `gateway_config` and `team_root` are supplied (notebook 01, §3). A minimal app like `app2` has neither, so a chat connection attempt should fail cleanly with "Chat is not enabled" rather than hanging or crashing.

In [7]:
with client.websocket_connect("/ws/chat/demo-agent") as ws:
    ws.send_text(json.dumps({"token": "walk-viewer"}))
    frame = ws.receive_json()
    print("chat on a minimal app:", frame)

chat on a minimal app: {'error': 'Chat is not enabled on this server'}


To get a live agent conversation, `create_app()` needs `team_root` pointing at a real `team/` directory and a `gateway_config` (SPEC-023) — the lifespan then calls `arcgateway.bootstrap.build_for_embedded`, which builds a real `WebPlatformAdapter` and wires it onto `app.state.web_adapter`. That's an integration-level setup outside this notebook's mock-first scope; see `packages/arcgateway/src/arcgateway/adapters/web.py` for the adapter itself.

The rest of this section demonstrates `/ws/team`, which needs no such setup — `team_stream` is always present.

In [8]:
# TeamStreamHub fan-out, exercised directly (not through a live WS) so we
# stay on the notebook's own event loop. Any hashable stands in for a
# WebSocket — the hub only ever uses it as a dict key.
import asyncio

from arcui.team_stream import TeamStreamHub, render_team_frame

hub = TeamStreamHub()
ws_a, ws_b = object(), object()
hub.register(ws_a)
hub.register(ws_b)

raw_message = {
    "sender": "did:arc:local:agent/intake",
    "to": ["channel://ops"],
    "body": "build finished",
    "mentions": [],
    "action_required": False,
    "priority": "normal",
    "seq": 1,
    "ts": "2026-05-08T12:00:00Z",
    "id": "msg-1",
}
frame = render_team_frame(raw_message)
print("rendered frame:", frame)

breakdown = await hub.publish(frame)
print("publish breakdown:", breakdown)
print("ws_a received:", await hub.next_frame(ws_a))
print("ws_b received:", await hub.next_frame(ws_b))

rendered frame: {'type': 'team_message', 'channel': 'ops', 'from': 'intake', 'body': 'build finished', 'mentions': [], 'action_required': False, 'priority': 'normal', 'seq': 1, 'ts': '2026-05-08T12:00:00Z', 'id': 'msg-1'}
publish breakdown: {'delivered': 2, 'dropped_backpressure': 0, 'dead': 0}
ws_a received: {'type': 'team_message', 'channel': 'ops', 'from': 'intake', 'body': 'build finished', 'mentions': [], 'action_required': False, 'priority': 'normal', 'seq': 1, 'ts': '2026-05-08T12:00:00Z', 'id': 'msg-1'}
ws_b received: {'type': 'team_message', 'channel': 'ops', 'from': 'intake', 'body': 'build finished', 'mentions': [], 'action_required': False, 'priority': 'normal', 'seq': 1, 'ts': '2026-05-08T12:00:00Z', 'id': 'msg-1'}


Both sockets received the frame — that is the fan-out guarantee. If a socket's queue is full the **oldest** frame is dropped to make room (`TeamStreamHub._enqueue_drop_oldest`, `queue_maxsize=100` by default) — a slow browser tab can never back up the publish loop or block another viewer. We exercise that in section 7.

## 5. Authentication on WS connections

The WebSocket auth is intentionally **not** the same as the REST auth. REST runs through ``AuthMiddleware`` and reads ``Authorization: Bearer <token>`` headers. WebSockets accept the connection first, then expect a JSON message containing the token within 5 seconds (``AUTH_TIMEOUT_SECONDS`` in ``arcui.ws_helpers``). If the message is malformed, missing, or the token is wrong, the server sends an error frame and closes the connection.

Two rules govern the WS auth path (``authenticate_ws``, shared by both routes):

1. **Auth before any data.** No frame reaches an unauthenticated client.
2. **Constant-time comparison.** Token validation goes through ``AuthConfig.validate_token``, which uses ``hmac.compare_digest`` to eliminate timing side channels.

Both routes additionally reject any role other than ``viewer``/``operator`` after auth succeeds (``role not in ("viewer", "operator")``) — a defense-in-depth check now that ``AuthConfig`` only issues those two roles in the first place.

In [9]:
# A bad token is rejected and the connection is closed.
with client.websocket_connect("/ws/team") as ws:
    ws.send_text(json.dumps({"token": "definitely-not-the-token"}))
    err = ws.receive_json()
    print("error frame:", err)

error frame: {'error': 'Invalid token'}


Malformed JSON, a missing `token` field, and a timeout all take the same path in `authenticate_ws` — the client gets a single `{"error": ...}` frame, then a close (`4001` for timeout/malformed, `4003` for a token that parses but doesn't validate). There is no distinction in the response between "you sent garbage" and "your token is wrong"; that information stays in the server log to avoid telling a probe exactly *why* their attempt failed.

In [10]:
# An operator token works the same way as a viewer, with a different role label.
with client.websocket_connect("/ws/team") as ws:
    ws.send_text(json.dumps({"token": "walk-op"}))
    resp = ws.receive_json()
    print("frame:", resp)
    assert resp["type"] == "ready"

frame: {'type': 'ready'}


The role isn't echoed back on the `ready` frame — the server just knows it internally as `request.state`-equivalent for the socket's lifetime. Both viewer and operator can read `/ws/team` and open `/ws/chat`; role-based authorization is enforced per-route where it matters, e.g. `PATCH /api/config` requires `role == "operator"` (notebook 01, §5's operator-only demo) — no `/api/agent` socket or agent role exists anymore to enforce anything against.

In [11]:
# Malformed first-message JSON is treated the same as a timeout — one
# error frame, then a 4001 close. No token comparison ever runs.
with client.websocket_connect("/ws/team") as ws:
    ws.send_text("not-json-at-all{{{")
    err = ws.receive_json()
    print("malformed first message:", err)

malformed first message: {'error': 'Auth timeout or invalid message'}


Same error shape, different close code (`4001` here vs. `4003` for a well-formed-but-wrong token) — the client can't tell the two apart from the frame content alone, only from the close code if it's inspecting one.

## 6. Channel-scoped subscription on `/ws/team`

There's no per-agent/per-layer `Subscription` filter model anymore — `/ws/team` has exactly one narrowing knob: the `?channel=<name>` query parameter on the connect URL. `TeamStreamHub.register(ws, channels=scope)` takes `scope` as either `None` (team-wide observer — every channel) or a `set[str]` of channel names to admit.

The match is on `frame["channel"]` (set by `render_team_frame`'s `_channel_of`, which reads the first `channel://` target in the raw message's `to` list). A socket registered with a scope only receives frames whose channel is in that scope; a socket registered with `channels=None` receives everything.

In [12]:
scoped_hub = TeamStreamHub()
ops_only, all_channels = object(), object()

scoped_hub.register(ops_only, channels={"ops"})  # scoped to one channel
scoped_hub.register(all_channels)  # channels=None default -> sees everything

ops_frame = render_team_frame(
    {"sender": "agent://intake", "to": ["channel://ops"], "body": "ops update", "seq": 1}
)
alerts_frame = render_team_frame(
    {"sender": "agent://intake", "to": ["channel://alerts"], "body": "alert!", "seq": 2}
)

await scoped_hub.publish(ops_frame)
await scoped_hub.publish(alerts_frame)

print("ops_only queue size    :", scoped_hub.queue_for(ops_only).qsize())
print("all_channels queue size:", scoped_hub.queue_for(all_channels).qsize())

ops_only queue size    : 1
all_channels queue size: 2


`ops_only` (scoped to `{"ops"}`) only saw the ops-channel frame; `all_channels` (unscoped) saw both. This is real server-side filtering — `TeamStreamHub.publish` checks each registered socket's scope before enqueueing, so a scoped viewer never even receives the bytes for a channel it isn't watching.

In [13]:
# Late-join replay: a socket registered AFTER frames were published still
# sees recent in-scope history, because `register()` replays the hub's
# bounded ring buffer (ring_maxlen=200 by default) before returning.
replay_hub = TeamStreamHub()
early_frame = render_team_frame(
    {"sender": "agent://intake", "to": ["channel://ops"], "body": "before you joined", "seq": 1}
)
await replay_hub.publish(early_frame)  # no sockets registered yet — only the ring keeps it

late_joiner = object()
replay_hub.register(late_joiner)  # synchronous — ring snapshot + registration are atomic

print("late joiner immediately has:", replay_hub.queue_for(late_joiner).qsize(), "frame(s) queued")
print("content:", await replay_hub.next_frame(late_joiner))

late joiner immediately has: 1 frame(s) queued
content: {'type': 'team_message', 'channel': 'ops', 'from': 'intake', 'body': 'before you joined', 'mentions': [], 'action_required': False, 'priority': 'normal', 'seq': 1, 'ts': '', 'id': ''}


This is what makes `/ws/team` useful for a tab opened mid-conversation — a late-joining viewer gets a snapshot of recent flow instead of staring at an empty panel until the next message happens to arrive. There is no equivalent replay for `/ws/chat`'s general flow, but `?since_seq=` (section 4) covers the specific reconnect case there.

In [14]:
# Scope a live connection to one channel via the query param, exercising
# the exact route code the browser uses on connect.
with client.websocket_connect("/ws/team?channel=ops") as ws:
    ws.send_text(json.dumps({"token": "walk-viewer"}))
    ws.receive_json()  # ready

    # Publish directly against the app's real hub — same object the route reads.
    matching = render_team_frame(
        {"sender": "agent://intake", "to": ["channel://ops"], "body": "matches", "seq": 1}
    )
    non_matching = render_team_frame(
        {"sender": "agent://intake", "to": ["channel://random"], "body": "filtered out", "seq": 2}
    )
    await app2.state.team_stream.publish(matching)
    await app2.state.team_stream.publish(non_matching)

    received = ws.receive_json()
    print("received over the wire:", received["channel"], "-", received["body"])

received over the wire: ops - matches


Both `/ws/chat` and `/ws/team` are typed, purpose-built protocols now rather than a generic filtered event bus — there's no separate "raw passthrough vs. filtered UIEvent" distinction to reason about. Each route owns its own frame shapes end-to-end.

## 7. Backpressure handling on slow clients

A real dashboard has at least one tab open on a laptop in a coffee shop with a flaky connection. If the browser stalls, where do the frames go?

`/ws/team`'s answer is `TeamStreamHub`'s per-socket bounded queue (`asyncio.Queue(maxsize=queue_maxsize)`, default `100`). `publish()` never blocks on a full queue — `_enqueue_drop_oldest` drops the single oldest queued frame and inserts the new one. A slow or dead browser tab can never stall the publish loop or block another viewer's queue.

There is no `EventBuffer`, no separate flush cadence, and no `ping` heartbeat frame anymore — those belonged to the deleted push pipeline. `/ws/chat` and `/ws/team` are both request/response-shaped enough (or, for `/ws/team`, backed by `TeamStreamHub`'s own bounded queue) that they don't need a second buffering layer in front.

`MAX_WS_MESSAGE_SIZE` (1 MB, in `arcui.ws_helpers`) is defined as a DoS-prevention constant but isn't currently enforced by either route's inbound frame handling — worth knowing if you're auditing message-size limits, since the constant existing doesn't mean the limit is wired in everywhere yet.

In [15]:
# Drop-oldest backpressure, exercised directly against TeamStreamHub's
# internal helper. A tiny queue_maxsize simulates a slow/backed-up client.
tiny_hub = TeamStreamHub(queue_maxsize=3)
slow_client = object()
tiny_hub.register(slow_client)
tiny_queue = tiny_hub.queue_for(slow_client)

for i in range(5):
    outcome = tiny_hub._enqueue_drop_oldest(tiny_queue, {"i": i})
    print(f"  enqueue i={i}: {outcome}")

drained = []
while not tiny_queue.empty():
    drained.append(tiny_queue.get_nowait())
print("survivors after overflow:", drained)

  enqueue i=0: delivered
  enqueue i=1: delivered
  enqueue i=2: delivered
  enqueue i=3: dropped_backpressure
  enqueue i=4: dropped_backpressure
survivors after overflow: [{'i': 2}, {'i': 3}, {'i': 4}]


The first two messages were evicted; the **most recent** three made it through. For a live-flow view this is the right tradeoff: an operator who steps away and comes back wants to see what's happening *now*, not the oldest backlog of stale frames.

In [16]:
# A giant post-auth message doesn't crash the connection — it just fails
# JSON parsing like any other malformed frame and gets a per-frame error
# reply. (MAX_WS_MESSAGE_SIZE exists as a constant but isn't currently
# enforced as a size guard on either route's inbound frame handling —
# this demonstrates the JSON-parse failure path, not a size check.)
#
# Fresh app/client here — app2's team_stream ring buffer already has
# frames queued from earlier cells (section 6), and a new connection
# would replay those before we get to send our own test frame.
from arcui.ws_helpers import MAX_WS_MESSAGE_SIZE

oversize_app = create_app(auth_config=auth)
oversize_client = TestClient(oversize_app)

with oversize_client.websocket_connect("/ws/team") as ws:
    ws.send_text(json.dumps({"token": "walk-viewer"}))
    ws.receive_json()  # ready

    oversized = "x" * (MAX_WS_MESSAGE_SIZE + 1)
    ws.send_text(oversized)
    err = ws.receive_json()
    print("oversized-but-malformed frame:", err["type"], err["code"])

    # Connection is still alive — publish and confirm delivery keeps working.
    still_alive = render_team_frame(
        {"sender": "agent://intake", "to": ["channel://ops"], "body": "still here", "seq": 3}
    )
    await oversize_app.state.team_stream.publish(still_alive)
    payload = ws.receive_json()
    print("after oversized frame, still receiving:", payload["body"])

oversized-but-malformed frame: error malformed
after oversized frame, still receiving: still here


A 1 MB+ frame that happens to be valid JSON of the expected shape (e.g. a very long `text`/`body` field) would currently be accepted and processed like any other frame — there's no size rejection at that layer today. If you're hardening a deployment against large-payload abuse, this is the gap to close (either in the route or upstream at a reverse proxy).

## 8. Cross-ref to arcllm's module stack

`attach_llm`'s entire job is discovering three module types from `arcllm.modules`: `CircuitBreakerModule`, `TelemetryModule`, `QueueModule`. The contract between the two packages is exactly:

```python
from arcllm import load_model
from arcui import attach_llm, create_app

app = create_app()
model = load_model('anthropic', telemetry=True)  # arcllm builds the module stack
attach_llm(app, model)                             # arcui discovers modules on it
```

The arcllm side (covered in `arcllm/09-telemetry-module.ipynb`) owns:

- Phase sub-timings (queue wait, network, deserialise, total)
- Token / cost accounting from response usage
- Writing records into the arcstore spool that `Observe` later reads

The arcui side (this notebook) owns:

- Discovering the attached modules for REST introspection (`/api/circuit-breakers`, `/api/budget`, `/api/queue`)
- The `/ws/chat` and `/ws/team` interactive surfaces
- Reading the durable record back out through `Observe` (notebook 01)

There is no live event relay between the two anymore — arcllm writes to the durable spool; arcui reads from it on demand. The split still makes both packages testable in isolation: arcllm tests don't need Starlette, arcui tests don't need an HTTP-mocked LLM provider.

In [17]:
# A demonstration of the contract — we import both ends and confirm the symbols exist.
import arcllm
from arcllm.modules import telemetry as arcllm_telemetry
import arcui

print("arcllm version:", getattr(arcllm, "__version__", "unknown"))
print("arcui version:", arcui.__version__)
print("arcllm TelemetryModule available:", hasattr(arcllm_telemetry, "TelemetryModule"))
print("arcui attach_llm available:", hasattr(arcui, "attach_llm"))

arcllm version: 0.7.0
arcui version: 0.2.0
arcllm TelemetryModule available: True
arcui attach_llm available: True


If you want to *modify* what a circuit breaker or budget check reports, the right place to look is the arcllm side — `attach_llm` will discover whatever module types you add to the recognized set. If you want to change how the interactive chat or team stream behave, that's arcui's job and lives entirely in the routes we exercised in sections 4–7.

In [18]:
# A quick reference: the real arcui symbols the two protocols in this
# notebook lean on.
from arcui import (
    attach_llm,
    create_app,
    serve,
)
from arcui.observe import Observe
from arcui.team_stream import TeamStreamHub, TeamBusObserver, render_team_frame
from arcui.ws_helpers import authenticate_ws

for sym in [
    attach_llm,
    create_app,
    serve,
    Observe,
    TeamStreamHub,
    TeamBusObserver,
    render_team_frame,
    authenticate_ws,
]:
    print(f"{sym.__name__:24s} from {sym.__module__}")

attach_llm               from arcui.server
create_app               from arcui.server
serve                    from arcui.server
Observe                  from arcui.observe
TeamStreamHub            from arcui.team_stream
TeamBusObserver          from arcui.team_stream
render_team_frame        from arcui.team_stream
authenticate_ws          from arcui.ws_helpers


These eight symbols are the entire surface area you need to reason about the interactive/live side of the dashboard. Three are public re-exports (`attach_llm`, `create_app`, `serve`); the rest are the modules the `/ws/chat` and `/ws/team` routes are built from.

In [19]:
# There is no separate "warm-start" bridge anymore — Observe.start() (run
# from create_app()'s lifespan, notebook 01 §3) backfills the mirror from
# the durable spool + WORM store directly, on every boot. Point data_dir
# at a directory with real arcstore data and the first browser to connect
# after a restart already sees a populated stats/traces panel — no
# separate warm_start() call, no trace_store to wire in.
data_app = create_app(data_dir=None)  # None = default arcstore data dir
print("observe mirror type:", type(data_app.state.observe).__name__)

observe mirror type: Observe


If you want to see the durable record populated end-to-end, see `arcllm/14-trace-store.ipynb` — it writes real trace records into the arcstore spool. Point this notebook's `data_dir` at the same location and `/api/traces` (notebook 01, §6) returns them.

## 9. Summary

- **`attach_llm` is a discovery walk, not a live relay.** It finds `CircuitBreakerModule` / `TelemetryModule` / `QueueModule` on an arcllm provider's `_inner` stack and exposes them on `app.state` for REST introspection. `label` is accepted but unused. There is no `on_event` callback, no `EventBuffer`, no `RollingAggregator`.
- **Two typed WebSocket protocols, not one generic event bus.** `/ws/chat/{agent_id}` is bidirectional interactive chat through `arcgateway`'s `WebPlatformAdapter` (needs `team_root` + `gateway_config`); `/ws/team` is a read-only team-flow stream (`TeamStreamHub`) with a narrow post-forward path, always present.
- **Shared first-message auth.** Both routes use `authenticate_ws`: 5 s timeout, `hmac.compare_digest` token validation, close `4001` (timeout/malformed) or `4003` (invalid token or wrong role).
- **Filtering is channel-scoped, not agent/layer/team-scoped.** `/ws/team`'s only knob is `?channel=<name>`; `TeamStreamHub` also replays a bounded ring buffer to late joiners.
- **Backpressure is `TeamStreamHub`'s bounded per-socket queue.** Drop-oldest on overflow (`_enqueue_drop_oldest`, `queue_maxsize=100` default) — a slow viewer never blocks the publish loop or another viewer. There's no separate buffer/flush-cadence layer and no heartbeat frame anymore.
- **Cross-package contract, unchanged in spirit.** arcllm still produces telemetry (now into the durable arcstore spool); arcui still reads it, just on demand through `Observe` instead of over a live push socket.

**Next:** see `arcui/01-dashboard-bringup.ipynb` for the matching two-token-auth and dashboard launch story (this notebook assumed an app was already created); see `arcllm/09-telemetry-module.ipynb` for the source-side event production and `arcllm/14-trace-store.ipynb` for the durable record this notebook's `Observe` reads from.

**API surface covered:**

- `arcui.create_app`, `arcui.attach_llm`, `arcui.serve`
- `arcui.auth.AuthConfig`
- `arcui.observe.Observe`
- `arcui.team_stream.TeamStreamHub`, `render_team_frame`, `TeamBusObserver`
- `arcui.ws_helpers.authenticate_ws`, `AUTH_TIMEOUT_SECONDS`, `MAX_WS_MESSAGE_SIZE`, `CLOSE_AUTH_TIMEOUT`, `CLOSE_AUTH_INVALID`
- WebSocket protocol: `token` first-frame auth; `/ws/chat` — `ready`/`message`/`?since_seq=`; `/ws/team` — `ready`/`team_message`/`post`/`?channel=`.